# Configuration

#### 1. Unity Catalog - Catalog, Schema & Table Names

In [0]:
# Catalog
catalog = "wyckoff_catalog"

# Schema 
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

# External storage 
storage_account = "wyckoffsac"
storage_container = "wyckoff-data"
storage_credential = "wyckoff_credentials"
external_location = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net"
catalog_managed_location = f"{external_location}/{catalog}"

# Table name
tables = {
    # Bronze
    "raw_ohlcv": f"{catalog}.{bronze_schema}.raw_ohlcv",
    # Silver
    "cleaned_ohlcv": f"{catalog}.{silver_schema}.cleaned_ohlcv",
    "technical_indicators": f"{catalog}.{silver_schema}.technical_indicators",
    "wyckoff_features": f"{catalog}.{silver_schema}.wyckoff_features",
    # Gold
    "phase_labels": f"{catalog}.{gold_schema}.phase_labels",
    "ml_features": f"{catalog}.{gold_schema}.ml_features",
    "model_metrics": f"{catalog}.{gold_schema}.model_metrics",
    "cnn_lstm_predictions": f"{catalog}.{gold_schema}.cnn_lstm_predictions",
    "event_study_results": f"{catalog}.{gold_schema}.event_study_results",
    "component_contribution_results": f"{catalog}.{gold_schema}.component_contribution_results",
}

#### 2. Shared constants, enums, table names, hyperparameters, and VN market rules.

In [0]:
from enum import IntEnum, Enum
from dataclasses import dataclass, field
from typing import List, Dict

In [0]:
class Exchange(str, Enum):
    HOSE = "HSX"    # Ho Chi Minh Stock Exchange
    HNX = "HNX"     # Hanoi Stock Exchange
    UPCOM = "UPCOM" # Unlisted Public Company Market

# Price-limit rules
price_limits= {
    Exchange.HOSE: 0.07,    
    Exchange.HNX: 0.10,     
    Exchange.UPCOM: 0.15,   
}

# Default exchanges
default_exchanges = [Exchange.HOSE, Exchange.HNX]

# Minimum average daily volume 
min_avg_daily_volume = 50000

# Ingestion batch size 
ingestion_batch_size = 20

# vnstock API rate limit
API_call_per_minute = 50
API_call_interval = 60.0 / API_call_per_minute

# Databricks secret scope 
secret_scope = "wyckoff_secrets"
Vnstock_API_key_name = "vnstockapi"

# Minimum shares per trade 
min_shares_per_trade = 100

# Inference batch size
inference_batch_size = 256


#### 3. Data split & windowing

In [0]:
train_start = "2015-01-01"
train_end = "2022-12-31"
val_start = "2023-01-01"
val_end = "2023-12-31"
test_start = "2024-01-01"
test_end = "2024-12-31"

# full data range
data_start = "2015-01-01"
data_end = "2024-12-31"

# sliding window
window_size = 90      # 3 months of trading days (khoảng 1 quý)
window_stride = 5     # stride of 5 

# price-limit filter: exclude windows where >50% of bars hit price limit
price_limit_threshold = 0.5

#### 4. Feature configuration

In [0]:
# OHLCV columns

ohlcv_columns = ["open", "high", "low", "close", "volume"]

# derived columns from silver cleaning
derived_columns = [
    "daily_return", "log_return", "typical_price","spread", "prev_close", "price_limit_hit",
]

# technical indicator columns
technical_indicator_columns = [
    "rsi_14", "rsi_28", "macd", "macd_signal", "macd_hist",
    "ema_12", "ema_26",
    "sma_20", "sma_50",
    "bb_upper", "bb_middle", "bb_lower", "bb_width", "bb_pct",
    "obv",
    "vwap_20",
    "atr_14",
    "volume_ratio",
    "adx_14", "plus_di", "minus_di",
    "stoch_k", "stoch_d",
]

# wyckoff-specific features
# boolean feature columns 
boolean_feature_columns = [
    "volume_climax_up", "volume_climax_down", "stopping_volume",
    "no_demand", "no_supply", "spring_signal", "upthrust_signal",
    "sos_signal", "sow_signal", "test_signal", "price_limit_hit",
]

volume_feature_names = {
    "volume", "obv", "vwap_20", "volume_ratio",
    "volume_climax_up", "volume_climax_down",
    "stopping_volume", "no_demand", "no_supply",
    "volume_trend", "volume_price_divergence",
}

wyckoff_feature_columns = [
    "price_spread", "close_position",
    "volume_climax_up", "volume_climax_down",
    "stopping_volume", "no_demand", "no_supply",
    "spring_signal", "upthrust_signal",
    "sos_signal", "sow_signal",
    "effort_vs_result",
    "test_signal",
    "support_level", "resistance_level",
    "trading_range_width", "trend_direction",
    "days_in_range", "volume_trend", "relative_position",
    "breakout_strength", "volume_price_divergence",
]

# all ML feature columns 
ml_feature_columns = (
    ohlcv_columns + derived_columns + technical_indicator_columns + wyckoff_feature_columns
)

# number of features
number_of_features = len(ml_feature_columns)

### 5. Wyckoff phase Enum

In [0]:
class WyckoffPhase(IntEnum):
    """10 Wyckoff phases: 5 Accumulation + 5 Distribution."""
    accum_A = 0   # Selling Climax + Automatic Rally
    accum_B = 1   # Secondary Test / Trading Range building
    accum_C = 2   # Spring / Shakeout
    accum_D = 3   # Sign of Strength + Last Point of Support
    accum_E = 4   # Markup begins - breakout above TR
    distrib_A = 5  # Buying Climax + Automatic Reaction
    distrib_B = 6  # Secondary Test of BC / TR building
    distrib_C = 7  # Upthrust After Distribution (UTAD)
    distrib_D = 8  # Sign of Weakness + Last Point of Supply
    distrib_E = 9  # Markdown begins - breakdown below TR

phase_names: Dict[int, str] = {p.value: p.name for p in WyckoffPhase}
number_of_phases: int = len(WyckoffPhase)
phase_name_list: List[str] = [phase_names[i] for i in range(number_of_phases)]

# groupings for trading signals
accumulation_phases = [
    WyckoffPhase.accum_A, WyckoffPhase.accum_B, WyckoffPhase.accum_C,
    WyckoffPhase.accum_D, WyckoffPhase.accum_E,
]
distribution_phases = [
    WyckoffPhase.distrib_A, WyckoffPhase.distrib_B, WyckoffPhase.distrib_C,
    WyckoffPhase.distrib_D, WyckoffPhase.distrib_E,
]
buy_phases = [WyckoffPhase.accum_D, WyckoffPhase.accum_E]
sell_phases = [WyckoffPhase.distrib_D, WyckoffPhase.distrib_E]


### 6. Model hyperparameters

In [0]:
@dataclass
class CNNLSTMConfig:
    """Hyperparameters for the CNN-LSTM model."""
    # input
    window_size: int = 90
    num_features: int = number_of_features
    num_classes: int = number_of_phases  

    # Multi-scale CNN
    cnn_filters: List[int] = field(default_factory=lambda: [32, 64, 128])
    cnn_kernels: List[int] = field(default_factory=lambda: [3, 5, 7])
    cnn_dropout: float = 0.4

    # BiLSTM
    lstm_hidden: int = 64
    lstm_layers: int = 2
    lstm_dropout: float = 0.4

    # Multi-Head Attention
    attention_heads: int = 4
    attention_dropout: float = 0.2

    # Classifier MLP
    mlp_hidden: int = 64
    mlp_dropout: float = 0.5

    # Training
    batch_size: int = 64
    learning_rate: float = 5e-4
    weight_decay: float = 1e-3
    label_smoothing: float = 0.1
    max_class_weight: float = 5.0
    max_epochs: int = 100
    early_stop_patience: int = 15
    grad_clip_norm: float = 1.0
    scheduler: str = "plateau"  # ReduceLROnPlateau

    # Data augmentation for rare classes
    augment_rare_classes: bool = True
    augment_max_target: int = 500      # 
    augment_jitter_std: float = 0.02   # Gaussian noise std

    # MLflow
    experiment_name: str = "/wyckoff_cnn_lstm_experiment"
    model_name: str = "wyckoff_cnn_lstm"


@dataclass
class XGBoostConfig:
    """Hyperparameters for the XGBoost baseline."""
    n_estimators: int = 500
    max_depth: int = 8
    learning_rate: float = 0.05
    subsample: float = 0.8
    colsample_bytree: float = 0.8
    objective: str = "multi:softprob"
    num_class: int = number_of_phases
    eval_metric: str = "mlogloss"
    early_stopping_rounds: int = 30


@dataclass
class LSTMBaselineConfig:
    """Hyperparameters for the plain LSTM baseline."""
    window_size: int = 90
    num_features: int = number_of_features
    num_classes: int = number_of_phases
    lstm_hidden: int = 128
    lstm_layers: int = 2
    lstm_dropout: float = 0.3
    mlp_hidden: int = 128
    mlp_dropout: float = 0.4
    batch_size: int = 64
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1
    max_class_weight: float = 5.0
    max_epochs: int = 100
    early_stop_patience: int = 15
    grad_clip_norm: float = 1.0


@dataclass
class CNNBaselineConfig:
    """Hyperparameters for the plain CNN baseline."""
    window_size: int = 90
    num_features: int = number_of_features
    num_classes: int = number_of_phases
    # Conv stack
    conv1_channels: int = 32
    conv1_kernel: int = 5
    conv2_channels: int = 64
    conv2_kernel: int = 3
    conv3_channels: int = 96
    conv3_kernel: int = 3
    pool_size: int = 2
    # Classification head
    mlp_hidden: int = 64
    mlp_dropout: float = 0.3
    # Training schedule 
    batch_size: int = 64
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1
    max_class_weight: float = 5.0
    max_epochs: int = 100
    early_stop_patience: int = 15
    grad_clip_norm: float = 1.0

# Experiment names
baselines_experiment_name = "/wyckoff_baselines_experiment"

# Model names for evaluation
model_names = ["random", "majority", "xgboost", "cnn", "lstm_h64", "lstm_h128", "cnn_lstm"]


#### 7. Wyckoff parameters

In [0]:
@dataclass
class WyckoffLabelingConfig:
    """Thresholds for the Wyckoff labeler."""
    # trend detection
    prior_trend_bars: int = 20        # Min bars SMA20 < SMA50 for downtrend
    prior_trend_decline: float = 0.15  # Or 15% price decline in lookback
    prior_trend_lookback: int = 60     # Lookback window for price decline

    # support / Resistance tolerance band
    sr_tolerance: float = 0.02  # 2% of support/resistance levels

    # Selling Climax (SC) / Buying Climax (BC)
    climax_spread_lookback: int = 20   # Widest spread in N bars
    climax_volume_threshold: float = 2.0  # Volume > N× average

    # Automatic Rally (AR) / Automatic Reaction
    ar_max_bars: int = 10              # AR must occur within N bars of SC

    # Phase B duration
    phase_b_min_bars: int = 20
    phase_b_max_bars: int = 80

    # Spring / UTAD
    spring_volume_ratio: float = 0.7   # Low volume on spring (< 0.7x avg)
    spring_recovery_bars: int = 3      # Close back above support within N bars

    # SOS / SOW
    sos_volume_ratio: float = 1.5      # Volume > 1.5x avg on SOS

    # Phase E: sustained breakout
    breakout_confirm_bars: int = 5     # Bars above/below TR to confirm Phase E

    # Confidence
    min_confidence: float = 0.3        # Minimum confidence to assign a label

#### 8. Event study parameters

In [0]:
event_study_list: List[int] = [5, 10, 20, 60]

#### 9. Visualization constants

In [0]:
# Phase color 
phase_colors = {
    WyckoffPhase.accum_A: "#1f77b4",    # blue
    WyckoffPhase.accum_B: "#aec7e8",    # light blue
    WyckoffPhase.accum_C: "#2ca02c",    # green
    WyckoffPhase.accum_D: "#98df8a",    # light green
    WyckoffPhase.accum_E: "#00ff00",    # bright green
    WyckoffPhase.distrib_A: "#ff7f0e",  # orange
    WyckoffPhase.distrib_B: "#ffbb78",  # light orange
    WyckoffPhase.distrib_C: "#d62728",  # red
    WyckoffPhase.distrib_D: "#ff9896",  # light red
    WyckoffPhase.distrib_E: "#ff0000",  # bright red
}

phase_colors_map = {p.value: c for p, c in phase_colors.items()}


### 10. Config Summary

In [0]:
def config_summary():
    """Print a summary of the configuration for verification."""
    print("Configuaration Summary")
    print(f"\nPhases: {number_of_phases} ({', '.join(phase_names.values())})")
    print(f"Exchanges: {[ex.value for ex in default_exchanges]}")
    for ex in default_exchanges:
        print(f"  {ex.value} price limit: ±{price_limits[ex]:.0%}")
    print(f"Min avg daily volume: {min_avg_daily_volume:,}")
    print(f"\nCatalog: {catalog}")
    print(f"Tables: {len(tables)}")
    for key, table in tables.items():
        print(f"  {key}: {table}")
    print(f"\nData range: {data_start} to {data_end}")
    print(f"Train: {train_start} to {train_end}")
    print(f"Val:   {val_start} to {val_end}")
    print(f"Test:  {test_start} to {test_end}")
    print(f"\nWindow: {window_size} bars, stride {window_stride}")
    print(f"ML features: {number_of_features} columns")

config_summary()
